In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import dotenv
from tqdm import tqdm
from openai import OpenAI
from mistralai import Mistral


from utils import load_json, extract_dialogue, dialogue_to_string, format_prompt, extract_xml, save_json
from prompts import system_prompt, mistake_prompt
from llm import llm_call

dotenv.load_dotenv()


False

In [3]:
dev_data_path = "data/source/mrbench_v3_devset.json"
test_data_path = "data/source/mrbench_v3_testset.json"
output_dir = "output_task1/"

In [4]:
dev_data = load_json(dev_data_path)
test_data = load_json(test_data_path)

len(dev_data), len(test_data)

(300, 191)

In [5]:
test_data[0]

{'conversation_id': '1030-adb61831-0383-4e51-a673-ab978590f69b',
 'conversation_history': 'Tutor: Hi, could you please provide a step-by-step solution for the question below? The question is: Tyson decided to make muffaletta sandwiches for the big game.  Each sandwich required 1 pound each of meat and cheese and would serve 4 people.  There would be 20 people in total watching the game.  The meat cost $7.00 per pound and the cheese cost $3.00 per pound.  How much money would he spend on the meat and cheese to make enough sandwiches to serve 20 people? \n Student: To serve 20 people, Tyson needs to make 20/4 = 5 sandwiches.\nEach sandwich requires 1+1 = 2 pounds of meat and cheese.\nFor 5 sandwiches, he needs a total of 2 x 5 = 10 pounds of meat and cheese.\nThe cost of 10 pounds of meat is 10 x $7.00 = $70.\nThe cost of 10 pounds of cheese is 10 x $3.00 = $30.\nThe total cost of meat and cheese is $70 + $30 = $100.\n 100 \n Tutor: do you want to talk me through your solution \n Student

In [35]:
already_processed_files = os.listdir(output_dir)
already_processed_files = [file.split(".")[0] for file in already_processed_files]

for example in test_data:
    if example['conversation_id'] in already_processed_files:
        continue
    dialogue = extract_dialogue(example["conversation_history"])
    dialogue_string = dialogue_to_string(dialogue)
    for tutor in tqdm(example['tutor_responses']):
        tutor_response = example['tutor_responses'][tutor]['response']
        # format the prompt
        prompt = format_prompt(mistake_prompt, dialogue=dialogue_string, feedback=tutor_response)
        # call the LLM
        llm_response = llm_call(prompt, backend="openai", model="gpt-4o-mini")
        analysis = extract_xml(llm_response, "analysis")
        mistake = extract_xml(llm_response, "mistake")
        # add annotations to example
        example['tutor_responses'][tutor]['annotation'] = {'Mistake_Identification': mistake, 'Analysis': analysis}
        # save the example as json file
    save_json(f"output_task1/{example['conversation_id']}.json", example)

1030-adb61831-0383-4e51-a673-ab978590f69b
['1030-adb61831-0383-4e51-a673-ab978590f69b']
862-a856b5c7-e9e4-4cfe-a527-c5daf716d206
['1030-adb61831-0383-4e51-a673-ab978590f69b']


  0%|          | 0/8 [00:00<?, ?it/s]

backend used openai - gpt-4o-mini


  0%|          | 0/8 [00:02<?, ?it/s]


KeyboardInterrupt: 

In [36]:
from pathlib import Path

output_dir_path = Path(output_dir)
already_processed_files = {p.stem for p in output_dir_path.glob("*.json")}

for example in tqdm(test_data, desc="Processing examples"):
    conv_id = example['conversation_id']
    if conv_id in already_processed_files:
        continue

    dialogue_string = dialogue_to_string(extract_dialogue(example["conversation_history"]))

    for tutor_id, tutor_info in example['tutor_responses'].items():
        tutor_response = tutor_info['response']

        prompt = format_prompt(
            mistake_prompt,
            dialogue=dialogue_string,
            feedback=tutor_response
        )

        llm_response = llm_call(prompt, backend="openai", model="gpt-4o-mini")

        tutor_info['annotation'] = {
            'Mistake_Identification': extract_xml(llm_response, "mistake"),
            'Analysis': extract_xml(llm_response, "analysis"),
        }

    save_json(output_dir_path / f"{conv_id}.json", example)

Processing examples:   0%|          | 0/191 [00:00<?, ?it/s]

backend used openai - gpt-4o-mini
backend used openai - gpt-4o-mini
backend used openai - gpt-4o-mini
backend used openai - gpt-4o-mini
backend used openai - gpt-4o-mini
backend used openai - gpt-4o-mini
backend used openai - gpt-4o-mini
backend used openai - gpt-4o-mini


Processing examples:   1%|          | 2/191 [00:27<43:15, 13.73s/it]

backend used openai - gpt-4o-mini


Processing examples:   1%|          | 2/191 [00:29<46:45, 14.84s/it]


KeyboardInterrupt: 

In [24]:
example

{'conversation_id': '1030-adb61831-0383-4e51-a673-ab978590f69b',
 'conversation_history': 'Tutor: Hi, could you please provide a step-by-step solution for the question below? The question is: Tyson decided to make muffaletta sandwiches for the big game.  Each sandwich required 1 pound each of meat and cheese and would serve 4 people.  There would be 20 people in total watching the game.  The meat cost $7.00 per pound and the cheese cost $3.00 per pound.  How much money would he spend on the meat and cheese to make enough sandwiches to serve 20 people? \n Student: To serve 20 people, Tyson needs to make 20/4 = 5 sandwiches.\nEach sandwich requires 1+1 = 2 pounds of meat and cheese.\nFor 5 sandwiches, he needs a total of 2 x 5 = 10 pounds of meat and cheese.\nThe cost of 10 pounds of meat is 10 x $7.00 = $70.\nThe cost of 10 pounds of cheese is 10 x $3.00 = $30.\nThe total cost of meat and cheese is $70 + $30 = $100.\n 100 \n Tutor: do you want to talk me through your solution \n Student

In [10]:
print(prompt)


You are an expert tutor. Your task is to analyze the student's answers and the tutor feedback determine whether the tutor was able to correctly identify the student's mistake in their final response.

Instructions:
- Read the conversation.
- Focus on the student's last response.
- Then read the tutor's final feedback.
- Decide if the tutor successfully identified the mistake made by the student.

Dialogue:
- Tutor: Hi, could you please provide a step-by-step solution for the question below? The question is: Tyson decided to make muffaletta sandwiches for the big game.  Each sandwich required 1 pound each of meat and cheese and would serve 4 people.  There would be 20 people in total watching the game.  The meat cost $7.00 per pound and the cheese cost $3.00 per pound.  How much money would he spend on the meat and cheese to make enough sandwiches to serve 20 people?
- Student: To serve 20 people, Tyson needs to make 20/4 = 5 sandwiches.
Each sandwich requires 1+1 = 2 pounds of meat an

In [11]:
print(llm_response)

<analysis> In the conversation, the student correctly identifies the number of sandwiches needed but miscalculates the total pounds of meat and cheese required. Each sandwich requires 1 pound of meat and 1 pound of cheese, meaning for 5 sandwiches, Tyson needs 5 pounds of meat and 5 pounds of cheese; therefore, the total is 10 pounds, but split between the two ingredients. The student then incorrectly assumes that all 10 pounds is for each category separately, resulting in the incorrect multiplication of $7 and $3 by 10. The tutor's feedback correctly points out the misuse of the quantities for meat and cheese, but fails to explicitly point out that the student should only calculate 5 pounds of meat and 5 pounds of cheese. Therefore, the tutor has identified that there is a mistake in the way the student has approached the multiplication, but did not clarify the specifics of their mistake leading to the erroneous total. </analysis>
<mistake> To some extent </mistake>


In [17]:
analysis = extract_xml(llm_response, "analysis")
mistake = extract_xml(llm_response, "mistake")

In [18]:
analysis, mistake

("In the conversation, the student correctly identifies the number of sandwiches needed but miscalculates the total pounds of meat and cheese required. Each sandwich requires 1 pound of meat and 1 pound of cheese, meaning for 5 sandwiches, Tyson needs 5 pounds of meat and 5 pounds of cheese; therefore, the total is 10 pounds, but split between the two ingredients. The student then incorrectly assumes that all 10 pounds is for each category separately, resulting in the incorrect multiplication of $7 and $3 by 10. The tutor's feedback correctly points out the misuse of the quantities for meat and cheese, but fails to explicitly point out that the student should only calculate 5 pounds of meat and 5 pounds of cheese. Therefore, the tutor has identified that there is a mistake in the way the student has approached the multiplication, but did not clarify the specifics of their mistake leading to the erroneous total.",
 'To some extent')

In [19]:
example

{'conversation_id': '1030-adb61831-0383-4e51-a673-ab978590f69b',
 'conversation_history': 'Tutor: Hi, could you please provide a step-by-step solution for the question below? The question is: Tyson decided to make muffaletta sandwiches for the big game.  Each sandwich required 1 pound each of meat and cheese and would serve 4 people.  There would be 20 people in total watching the game.  The meat cost $7.00 per pound and the cheese cost $3.00 per pound.  How much money would he spend on the meat and cheese to make enough sandwiches to serve 20 people? \n Student: To serve 20 people, Tyson needs to make 20/4 = 5 sandwiches.\nEach sandwich requires 1+1 = 2 pounds of meat and cheese.\nFor 5 sandwiches, he needs a total of 2 x 5 = 10 pounds of meat and cheese.\nThe cost of 10 pounds of meat is 10 x $7.00 = $70.\nThe cost of 10 pounds of cheese is 10 x $3.00 = $30.\nThe total cost of meat and cheese is $70 + $30 = $100.\n 100 \n Tutor: do you want to talk me through your solution \n Student